# reduce-gather-sum — ex1: compare reduce vs gather + manual sum

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reduce-gather-sum`. Running the final beacon cell reports progress against the `Distributed: reduce.gather + sum` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: reduce.gather + sum` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reduce-gather-sum`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reduce-gather-sum"
DD_SUBTOPIC = "Distributed: reduce.gather + sum"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. Each rank runs the same function in its own process; collectives operate in-place on tensors of identical shape across ranks.

**Backends.** `'nccl'` for multi-GPU (ARENA's setup), `'gloo'` for CPU (what these drills use — Colab CPU runtimes have no GPUs).

**Launch pattern.** Each test uses `mp.get_context('fork').Process` so worker fns defined in a notebook cell are picklable. Workers init the group, do their work, push results onto a `manager.Queue`, then destroy the group.

### This drill's atom: `reduce` vs `gather`
Both collect tensors from all ranks to ONE destination rank — but they do different things with them:

**`dist.reduce(tensor, dst=0, op=SUM)`** — every rank contributes its tensor; the destination rank ends with the *combined* value (sum, max, etc.). On non-dst ranks, the tensor is left in an **implementation-defined state** (it may hold partial sums from the tree-reduction; do NOT use the value). Memory: dst holds ONE tensor.

**`dist.gather(tensor, gather_list=[...], dst=0)`** — every rank contributes its tensor; the destination rank ends with a *list* of all N tensors (one per rank, in rank order). Memory: dst holds N tensors. From there you can `sum`, `mean`, take min, take median — anything `reduce` can't.

**Rule of thumb.** Use `reduce` if you only need the aggregate (and only consume the value on the dst rank). Use `gather` if you need the per-rank values (e.g., to compute a median, or log per-rank loss separately). `gather` uses N× the memory.

### Exercise 1 — compare reduce vs gather + manual sum

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Apply both `dist.reduce(SUM)` and `dist.gather` + manual list sum at rank 0 to confirm they produce the same total but differ in what intermediate values are accessible.
> Keywords: reduce, gather, dst, gather_list, aggregation
> ```

**KCs targeted:** `reduce-collapses-to-aggregate`, `gather-preserves-per-rank-values`

Implement `ex1_reduce_vs_gather_worker(rank, world_size, port, out_queue)`. Each rank starts with `tensor = t.tensor([float(rank + 1)])`.

On EVERY rank, do BOTH collective patterns sequentially:

**Path A — `reduce`:**
```python
a = t.tensor([float(rank + 1)])
dist.reduce(a, dst=0, op=dist.ReduceOp.SUM)
# rank 0 now holds the SUM; other ranks hold their ORIGINAL value
```

**Path B — `gather` + manual sum:**
```python
b = t.tensor([float(rank + 1)])
if rank == 0:
    gather_list = [t.zeros(1) for _ in range(world_size)]
else:
    gather_list = None
dist.gather(b, gather_list=gather_list, dst=0)
if rank == 0:
    gathered_sum = sum(g.item() for g in gather_list)
    per_rank = [g.item() for g in gather_list]
else:
    gathered_sum = None
    per_rank = None
```

Push `(rank, a.item(), gathered_sum, per_rank)` onto `out_queue`. On rank>0, `gathered_sum` and `per_rank` will be `None`.

**Expected (world_size=3, inputs [1,2,3]).** Rank 0: `a=6.0` (the sum), `gathered_sum=6.0`, `per_rank=[1.0,2.0,3.0]`. On non-dst ranks (1, 2) the `a` value is implementation-defined garbage — the test does NOT assert what it is, only that it is NOT silently the sum (which would mean you accidentally used `all_reduce`).

In [ ]:
import os, datetime
import torch as t
import torch.distributed as dist

def ex1_reduce_vs_gather_worker(rank, world_size, port, out_queue):
    """Init gloo, run BOTH reduce and gather on the same tensor, queue both outcomes."""
    raise NotImplementedError()


def _test_ex1():
    import os as _os
    import datetime as _dt
    import torch.distributed as _dist
    import torch.multiprocessing as _mp

    def _dd_run_workers(worker_fn, world_size, port, *extra_args, timeout=30):
        """Spawn `world_size` fork-context procs, return list of exitcodes."""
        ctx = _mp.get_context('fork')
        procs = []
        for rank in range(world_size):
            p = ctx.Process(target=worker_fn, args=(rank, world_size, port, *extra_args))
            p.start()
            procs.append(p)
        for p in procs:
            p.join(timeout=timeout)
        codes = [p.exitcode for p in procs]
        for p in procs:
            if p.is_alive():
                p.terminate()
        return codes

    manager = _mp.Manager()
    q = manager.Queue()
    codes = _dd_run_workers(ex1_reduce_vs_gather_worker, 3, 29616, q)
    assert codes == [0, 0, 0], f'workers failed: {codes}'

    results = {}
    while not q.empty():
        rank, a_val, gathered_sum, per_rank = q.get()
        results[rank] = (a_val, gathered_sum, per_rank)

    assert set(results.keys()) == {0, 1, 2}

    # --- Rank 0: reduce result + gather list + manual sum ---
    a0, gs0, pr0 = results[0]
    assert abs(a0 - 6.0) < 1e-6, f'rank 0 reduce: expected 6.0, got {a0}'
    assert gs0 is not None and abs(gs0 - 6.0) < 1e-6, (
        f'rank 0 gather sum: expected 6.0, got {gs0}'
    )
    assert pr0 == [1.0, 2.0, 3.0], (
        f'rank 0 per_rank list: expected [1.0, 2.0, 3.0], got {pr0} '
        f'(gather preserves per-rank values in rank order)'
    )

    # --- Rank 1, rank 2: reduce leaves non-dst tensors in an
    # implementation-defined state. We DON'T assert a specific value;
    # we ONLY assert the rank still ran (got past reduce) and that
    # gather metadata is None on non-dst ranks (key contrast vs all_*).
    for non_dst in (1, 2):
        a_v, gs_v, pr_v = results[non_dst]
        assert isinstance(a_v, float), (
            f'rank {non_dst} reduce: expected a scalar float (non-dst value is\n'
            f'implementation-defined, but the call must still complete), got {a_v!r}'
        )
        assert gs_v is None, (
            f'rank {non_dst} gather sum should be None — only dst rank computes the sum '
            f'(if not None, you ran the sum on all ranks → that\'s all_gather behavior, not gather)'
        )
        assert pr_v is None, f'rank {non_dst} per_rank should be None, got {pr_v!r}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_reduce_vs_gather_worker(rank, world_size, port, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    # --- Path A: reduce collapses to aggregate at dst=0 only ---
    a = t.tensor([float(rank + 1)])
    dist.reduce(a, dst=0, op=dist.ReduceOp.SUM)
    # --- Path B: gather preserves per-rank values at dst=0 only ---
    b = t.tensor([float(rank + 1)])
    if rank == 0:
        gather_list = [t.zeros(1) for _ in range(world_size)]
    else:
        gather_list = None
    dist.gather(b, gather_list=gather_list, dst=0)
    if rank == 0:
        gathered_sum = sum(g.item() for g in gather_list)
        per_rank = [g.item() for g in gather_list]
    else:
        gathered_sum = None
        per_rank = None
    out_queue.put((rank, a.item(), gathered_sum, per_rank))
    dist.destroy_process_group()
```

**Key insight: only dst is guaranteed.** Beginners often assume every rank gets the sum after `reduce` — that's `all_reduce`, not `reduce`. With `reduce`, ONLY the dst rank's tensor is guaranteed to hold the aggregate. Non-dst ranks may hold their original value, a partial sum from the tree-reduction, or whatever the backend's implementation produced. **Never read the non-dst tensor.** This drill enforces that: the test does not assert the non-dst value, only that the call returned a scalar.

**Different backends, different garbage.** On `gloo` you'll often see tree-reduction partials on non-dst ranks (e.g. for world_size=3 with inputs [1,2,3], rank 1 might end with 5.0 = 1+(2*2) due to pairwise reduce). On `nccl` you may see the original value untouched. Both are correct per the API contract.

**When to choose `gather`.** Any aggregation that's not a binary op: median, percentile, min-with-rank-tag, top-k. The trade-off is memory: gather_list at dst holds N tensors; reduce holds 1.

**Symmetric variants.** `all_reduce` = reduce-to-everyone (the sum IS guaranteed on all ranks). `all_gather` = gather-to-everyone (each rank ends with the same length-N list). Pick based on whether downstream code needs the result on every rank or just on rank 0.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()